In [1]:
# Install the libraries needed for our Mini RAG Chatbot.

!pip -q install sentence-transformers numpy

In [2]:
# Sample document containing scholarship information.
# Our chatbot will search this text to answer questions.

documents = [
    """
    The Global Excellence Scholarship supports international students
    pursuing postgraduate degrees.

    Applicants must hold a recognized bachelor's degree and demonstrate
    strong academic performance.

    Applicants must submit academic transcripts, a statement of purpose,
    recommendation letters, and proof of English language proficiency.
    """,

    """
    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants should check the official scholarship website for the
    application deadline and any additional requirements.

    Incomplete applications may not be considered.
    """,

    """
    Scholarship recipients may receive financial support toward tuition
    fees and living expenses.

    The amount of funding depends on the specific scholarship award.
    Applicants should consult the official scholarship documentation
    for the exact amount and coverage.
    """
]

print("Number of documents:", len(documents))
print("First document:\n", documents[0])

Number of documents: 3
First document:
 
    The Global Excellence Scholarship supports international students
    pursuing postgraduate degrees.

    Applicants must hold a recognized bachelor's degree and demonstrate
    strong academic performance.

    Applicants must submit academic transcripts, a statement of purpose,
    recommendation letters, and proof of English language proficiency.
    


In [3]:
# Import the embedding model.
from sentence_transformers import SentenceTransformer

# Load a pretrained model that converts text into embeddings.
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert every document into a numerical vector.
document_embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", document_embeddings.shape)
print("Number of document embeddings:", len(document_embeddings))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (3, 384)
Number of document embeddings: 3


In [72]:
import numpy as np

def retrieve_documents(question, top_k=3):

    # Convert the question into an embedding
    question_embedding = model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculate similarity with each document
    similarities = document_embeddings @ question_embedding

    question_lower = question.lower()

    # Identify the topic of the question
    eligibility_keywords = [
        "eligible", "eligibility", "qualification",
        "qualifications", "bachelor", "academic performance",
        "who can apply", "who is eligible"
    ]

    application_keywords = [
        "deadline", "application process", "how to apply",
        "application form", "submit documents", "submission"
    ]

    funding_keywords = [
        "funding", "scholarship amount", "how much",
        "money", "tuition fees", "living expenses"
    ]

    # Identify the most relevant document for the topic
    target_index = None

    if any(word in question_lower for word in eligibility_keywords):
        target_index = 0

    elif any(word in question_lower for word in application_keywords):
        target_index = 1

    elif any(word in question_lower for word in funding_keywords):
        target_index = 2

    # Copy similarity scores so the original scores remain unchanged
    final_scores = similarities.copy()

    # Give the topic's document a ranking boost
    if target_index is not None:
        final_scores[target_index] += 0.30

    # Rank documents using adjusted scores
    ranked_indices = np.argsort(final_scores)[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        results.append({
            "document": documents[index],
            "similarity": float(similarities[index])
        })

    return results

print("Retrieval function updated successfully!")

Retrieval function updated successfully!


In [73]:
results = retrieve_documents(
    "Who is eligible to apply for the scholarship?",
    top_k=3
)

for i, result in enumerate(results, start=1):
    print(f"\nSource {i} — Similarity: {result['similarity']:.3f}")
    print(result["document"])


Source 1 — Similarity: 0.626

    The Global Excellence Scholarship supports international students
    pursuing postgraduate degrees.

    Applicants must hold a recognized bachelor's degree and demonstrate
    strong academic performance.

    Applicants must submit academic transcripts, a statement of purpose,
    recommendation letters, and proof of English language proficiency.
    

Source 2 — Similarity: 0.631

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants should check the official scholarship website for the
    application deadline and any additional requirements.

    Incomplete applications may not be considered.
    

Source 3 — Similarity: 0.591

    Scholarship recipients may receive financial support toward tuition
    fees and living expenses.

    The amount of funding depends on the specific scholarship award.
    Applicants should consult the official scholarship docu

In [50]:
from google.colab import userdata
userdata.get('GEMINI_API_KEY')

'AQ.Ab8RN6JCgVdHmp8oxXE_q0GwbkyLbSi0Ximz9S5ai3o5nSy46Q'

In [22]:
from google import genai
from google.colab import userdata

# Get the actual API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

# Create the Gemini client using the API key
client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [10]:
!pip -q install google-genai

In [23]:
from google import genai
from google.colab import userdata

# Load your API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

# Create the Gemini AI client
client = genai.Client(api_key=api_key)

print("Gemini AI connected successfully!")

Gemini AI connected successfully!


In [51]:
def generate_answer(question, top_k=2):

    # STEP 1: Retrieve the most relevant documents
    results = retrieve_documents(question, top_k=top_k)

    # STEP 2: Combine the retrieved documents into context
    context = "\n\n".join(
        f"Source {i}:\n{result['document']}"
        for i, result in enumerate(results, start=1)
    )

    # STEP 3: Create a prompt for Gemini
    prompt = f"""
    You are a scholarship information assistant.

    Answer the user's question using only the information
    provided in the context below.

    If the context does not contain the answer, say that
    the available documents do not provide that information.

    Do not invent scholarship requirements or deadlines.

    CONTEXT:
    {context}

    USER QUESTION:
    {question}

    Provide a clear and concise answer.
    """

    # STEP 4: Send the prompt to Gemini
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    # STEP 5: Return the answer and supporting documents
    return {
        "answer": response.text,
        "sources": results
    }

print("RAG answer-generation function created successfully!")

RAG answer-generation function created successfully!


In [53]:

def generate_answer(question, top_k=3):

    # STEP 1: Retrieve the most relevant documents
    results = retrieve_documents(question, top_k=top_k)

    # STEP 2: Combine retrieved documents into context
    context = "\n\n".join(
        f"Source {i}:\n{result['document']}"
        for i, result in enumerate(results, start=1)
    )

    # STEP 3: Create a prompt for Gemini
    prompt = f"""
You are a scholarship information assistant.

Answer the user's question using only the information
provided in the context below.

Instructions:
- Give a clear and concise answer.
- Do not invent scholarship requirements or deadlines.
- If the context does not contain the answer, clearly say
  that the available documents do not provide that information.
- Use the retrieved documents as your source of information.

CONTEXT:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    # STEP 4: Generate an answer using Gemini
    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    # STEP 5: Return the answer and supporting sources
    return {
        "answer": response.text,
        "sources": results
    }


print("RAG answer-generation function updated successfully!")



RAG answer-generation function updated successfully!


In [39]:
questions = [
    "What documents are required to apply for the scholarship?",
    "Who is eligible to apply for the scholarship?",
    "What is the minimum academic qualification required?",
    "What is the scholarship application deadline?",
    "Does the scholarship require English language proficiency?",
    "What happens if an application is incomplete?",
    "Who can apply for the Global Excellence Scholarship?",
    "What documents should applicants submit?"
]

for i, question in enumerate(questions, start=1):
    print("\n" + "=" * 70)
    print(f"QUESTION {i}: {question}")
    print("=" * 70)

    try:
        response = generate_answer(question)

        print("\nAI-GENERATED ANSWER:")
        print(response["answer"])

        print("\nSUPPORTING SOURCES:")

        for j, source in enumerate(response["sources"], start=1):
            print(f"\nSource {j}")
            print(f"Similarity Score: {source['similarity']:.3f}")
            print(source["document"])
            print("-" * 50)

    except Exception as e:
        print(f"Error processing this question: {e}")

print("\nQUESTION TESTING COMPLETED!")


QUESTION 1: What documents are required to apply for the scholarship?
Error processing this question: 'Model' object has no attribute 'encode'

QUESTION 2: Who is eligible to apply for the scholarship?
Error processing this question: 'Model' object has no attribute 'encode'

QUESTION 3: What is the minimum academic qualification required?
Error processing this question: 'Model' object has no attribute 'encode'

QUESTION 4: What is the scholarship application deadline?
Error processing this question: 'Model' object has no attribute 'encode'

QUESTION 5: Does the scholarship require English language proficiency?
Error processing this question: 'Model' object has no attribute 'encode'

QUESTION 6: What happens if an application is incomplete?
Error processing this question: 'Model' object has no attribute 'encode'

QUESTION 7: Who can apply for the Global Excellence Scholarship?
Error processing this question: 'Model' object has no attribute 'encode'

QUESTION 8: What documents should ap

In [41]:

from sentence_transformers import SentenceTransformer

# Reload the Sentence Transformers model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model reloaded successfully!")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model reloaded successfully!


In [42]:
question = "What documents are required to apply for the scholarship?"

results = retrieve_documents(question, top_k=2)

for i, result in enumerate(results, start=1):
    print(f"\nSource {i}")
    print(f"Similarity score: {result['similarity']:.3f}")
    print(result["document"])


Source 1
Similarity score: 0.662

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants should check the official scholarship website for the
    application deadline and any additional requirements.

    Incomplete applications may not be considered.
    

Source 2
Similarity score: 0.546

    The Global Excellence Scholarship supports international students
    pursuing postgraduate degrees.

    Applicants must hold a recognized bachelor's degree and demonstrate
    strong academic performance.

    Applicants must submit academic transcripts, a statement of purpose,
    recommendation letters, and proof of English language proficiency.
    


In [45]:
question = "What is the exact amount of money awarded by the scholarship?"

response = generate_answer(question)

print("QUESTION:")
print(question)

print("\nAI-GENERATED ANSWER:")
print(response["answer"])

print("\nRETRIEVED SOURCES:")
for i, source in enumerate(response["sources"], start=1):
    print(f"\nSource {i} | Similarity: {source['similarity']:.3f}")
    print(source["document"])

QUESTION:
What is the exact amount of money awarded by the scholarship?

AI-GENERATED ANSWER:
Based on the provided documents, the exact amount of funding depends on the specific scholarship award. Applicants should consult the official scholarship documentation for the exact amount and coverage, as the available documents do not state the exact amount.

RETRIEVED SOURCES:

Source 1 | Similarity: 0.628

    Scholarship recipients may receive financial support toward tuition
    fees and living expenses.

    The amount of funding depends on the specific scholarship award.
    Applicants should consult the official scholarship documentation
    for the exact amount and coverage.
    

Source 2 | Similarity: 0.506

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants should check the official scholarship website for the
    application deadline and any additional requirements.

    Incomplete app

In [46]:
question = "What is the exact amount of money awarded by the scholarship?"

response = generate_answer(question)

print("QUESTION:")
print(question)

print("\nAI-GENERATED ANSWER:")
print(response["answer"])

print("\nSUPPORTING SOURCES:")

for i, source in enumerate(response["sources"], start=1):
    print(f"\nSource {i}")
    print(f"Similarity Score: {source['similarity']:.3f}")
    print(source["document"])

QUESTION:
What is the exact amount of money awarded by the scholarship?

AI-GENERATED ANSWER:
The available documents do not provide that information. The exact amount of funding depends on the specific scholarship award, and applicants should consult the official scholarship documentation for the exact amount and coverage.

SUPPORTING SOURCES:

Source 1
Similarity Score: 0.628

    Scholarship recipients may receive financial support toward tuition
    fees and living expenses.

    The amount of funding depends on the specific scholarship award.
    Applicants should consult the official scholarship documentation
    for the exact amount and coverage.
    

Source 2
Similarity Score: 0.506

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants should check the official scholarship website for the
    application deadline and any additional requirements.

    Incomplete applications may not be 

In [47]:
question = "Who is eligible to apply for the scholarship?"

response = generate_answer(question, top_k=3)

print("QUESTION:")
print(question)

print("\nAI-GENERATED ANSWER:")
print(response["answer"])

print("\nSUPPORTING SOURCES:")

for i, source in enumerate(response["sources"], start=1):
    print(f"\nSource {i}")
    print(f"Similarity Score: {source['similarity']:.3f}")
    print(source["document"])

QUESTION:
Who is eligible to apply for the scholarship?

AI-GENERATED ANSWER:
Based on the provided documents, eligibility is specified for the Global Excellence Scholarship, which supports international students pursuing postgraduate degrees. Applicants must hold a recognized bachelor's degree and demonstrate strong academic performance.

SUPPORTING SOURCES:

Source 1
Similarity Score: 0.670

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants should check the official scholarship website for the
    application deadline and any additional requirements.

    Incomplete applications may not be considered.
    

Source 2
Similarity Score: 0.614

    Scholarship recipients may receive financial support toward tuition
    fees and living expenses.

    The amount of funding depends on the specific scholarship award.
    Applicants should consult the official scholarship documentation
    for the 

In [48]:
question = "Who is eligible to apply for the scholarship?"

for k in [2, 3]:
    print("\n" + "=" * 50)
    print(f"RETRIEVING TOP {k} DOCUMENTS")
    print("=" * 50)

    results = retrieve_documents(question, top_k=k)

    for i, result in enumerate(results, start=1):
        print(f"\nSource {i}")
        print(f"Similarity: {result['similarity']:.3f}")
        print(result["document"][:150] + "...")


RETRIEVING TOP 2 DOCUMENTS

Source 1
Similarity: 0.670

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants sho...

Source 2
Similarity: 0.614

    Scholarship recipients may receive financial support toward tuition
    fees and living expenses.

    The amount of funding depends on the speci...

RETRIEVING TOP 3 DOCUMENTS

Source 1
Similarity: 0.670

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants sho...

Source 2
Similarity: 0.614

    Scholarship recipients may receive financial support toward tuition
    fees and living expenses.

    The amount of funding depends on the speci...

Source 3
Similarity: 0.558

    The Global Excellence Scholarship supports international students
    pursuing postgraduate degrees.

    Applicants must hold a recognized bache...


In [44]:
questions = [
    "What documents are required to apply for the scholarship?",
    "Who is eligible to apply for the scholarship?",
    "What is the minimum academic qualification required?",
    "What is the scholarship application deadline?",
    "Does the scholarship require English language proficiency?",
    "What happens if an application is incomplete?",
    "Who can apply for the Global Excellence Scholarship?",
    "What documents should applicants submit?"
]

for i, question in enumerate(questions, start=1):

    print("\n" + "=" * 60)
    print(f"QUESTION {i}: {question}")
    print("=" * 60)

    try:
        response = generate_answer(question)

        print("\nAI-GENERATED ANSWER:")
        print(response["answer"])

        print("\nSUPPORTING SOURCES:")

        for j, source in enumerate(response["sources"], start=1):
            print(f"\nSource {j}")
            print(f"Similarity Score: {source['similarity']:.3f}")
            print(source["document"])

    except Exception as e:
        print(f"Error: {e}")

print("\nALL QUESTIONS TESTED!")


QUESTION 1: What documents are required to apply for the scholarship?

AI-GENERATED ANSWER:
Based on the provided documents, applicants for the Global Excellence Scholarship must submit the following documents:

* Academic transcripts
* A statement of purpose
* Recommendation letters
* Proof of English language proficiency

Additionally, the scholarship application process involves submitting "all required documents" along with completing an online application form, and applicants should check the official scholarship website for any additional requirements.

SUPPORTING SOURCES:

Source 1
Similarity Score: 0.662

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants should check the official scholarship website for the
    application deadline and any additional requirements.

    Incomplete applications may not be considered.
    

Source 2
Similarity Score: 0.546

    The Global Excellence Sc

In [55]:
!pip install -q gradio

In [58]:
import gradio as gr
def chat_response(message, history):
    if not message.strip():
        return "Please enter a question."

    # Collect previous conversation messages
    conversation_history = ""

    for item in history:
        if isinstance(item, dict):
            role = item.get("role", "")
            content = item.get("content", "")

            if role in ["user", "assistant"] and isinstance(content, str):
                conversation_history += f"{role}: {content}\n"

        elif isinstance(item, (list, tuple)) and len(item) == 2:
            conversation_history += (
                f"user: {item[0]}\nassistant: {item[1]}\n"
            )

    # Understand follow-up questions
    contextual_question = message

    if conversation_history:
        prompt = f"""
Previous conversation:
{conversation_history}

Current question:
{message}

Rewrite the current question so it can be understood
without the previous conversation.

Return only the rewritten question.
"""

        rewrite_response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt
        )

        contextual_question = rewrite_response.text.strip()

    # Generate an answer using the RAG system
    response = generate_answer(contextual_question)

    answer = response["answer"]
    answer += "\n\n### Supporting Sources\n"

    for i, source in enumerate(response["sources"], start=1):
        answer += (
            f"\n**Source {i}** "
            f"(Similarity: {source['similarity']:.3f})\n\n"
            f"{source['document']}\n\n"
        )

    return answer


print("Chat function updated successfully!")

Chat function updated successfully!


In [62]:
import gradio as gr

custom_css = """
.gradio-container {
    max-width: 1000px !important;
    margin: auto !important;
    font-family: Arial, sans-serif;
}

footer {
    display: none !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css
) as chat_demo:

    gr.Markdown(
        """
        # 🎓 Mini RAG Scholarship Chatbot
        ### Ask questions about eligibility, applications, and funding.
        """
    )

    gr.ChatInterface(
        fn=chat_response,
        chatbot=gr.Chatbot(
            height=480,
            show_label=False
        ),
        textbox=gr.Textbox(
            placeholder="Ask a scholarship question...",
            label="Your Question",
            lines=2
        ),
        examples=[
            "Who is eligible for the scholarship?",
            "What documents are required?",
            "What is the scholarship deadline?",
            "What is the exact scholarship amount?"
        ]
    )

chat_demo.launch(share=True)

/tmp/ipykernel_1111/3070372601.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://67cc6f45a6083dc6f7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [71]:
results = retrieve_documents(
    "Who is eligible to apply for the scholarship?",
    top_k=3
)

for i, result in enumerate(results, start=1):
    print(
        f"\nSource {i} — "
        f"Similarity: {result['similarity']:.3f}"
    )
    print(result["document"])


Source 1 — Similarity: 0.631

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants should check the official scholarship website for the
    application deadline and any additional requirements.

    Incomplete applications may not be considered.
    

Source 2 — Similarity: 0.626

    The Global Excellence Scholarship supports international students
    pursuing postgraduate degrees.

    Applicants must hold a recognized bachelor's degree and demonstrate
    strong academic performance.

    Applicants must submit academic transcripts, a statement of purpose,
    recommendation letters, and proof of English language proficiency.
    

Source 3 — Similarity: 0.591

    Scholarship recipients may receive financial support toward tuition
    fees and living expenses.

    The amount of funding depends on the specific scholarship award.
    Applicants should consult the official scholarship docu

In [65]:
# Add topic descriptions to help the retrieval system
search_documents = [
    """
    Topic: Scholarship eligibility and requirements.

    The Global Excellence Scholarship supports international students
    pursuing postgraduate degrees.

    Eligibility requirements:
    Applicants must hold a recognized bachelor's degree and
    demonstrate strong academic performance.

    Required documents include academic transcripts, a statement
    of purpose, recommendation letters, and proof of English
    language proficiency.
    """,

    """
    Topic: Scholarship application process and deadline.

    Applicants must complete an online scholarship application form
    and submit all required documents.

    Applicants should check the official scholarship website
    for the application deadline and additional requirements.

    Incomplete applications may not be considered.
    """,

    """
    Topic: Scholarship funding, financial support, and award amount.

    Scholarship recipients may receive financial support toward
    tuition fees and living expenses.

    The funding amount depends on the specific scholarship award.
    Applicants should consult the official scholarship documentation
    for the exact amount and coverage.
    """
]

# Create embeddings from the improved search documents
document_embeddings = model.encode(
    search_documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Improved search documents and embeddings created!")

Improved search documents and embeddings created!


In [78]:
test_questions = [
"Who is eligible to apply for the scholarship?",
"What is the application deadline?",
"What is the exact scholarship amount?"
]

for question in test_questions:
  print("\n" + "=" * 60)
  print("QUESTION:", question)

  results = retrieve_documents(question, top_k=1)

  print("\nTOP RETRIEVED DOCUMENT:")
  print(results[0]["document"])




QUESTION: Who is eligible to apply for the scholarship?

TOP RETRIEVED DOCUMENT:

    The Global Excellence Scholarship supports international students
    pursuing postgraduate degrees.

    Applicants must hold a recognized bachelor's degree and demonstrate
    strong academic performance.

    Applicants must submit academic transcripts, a statement of purpose,
    recommendation letters, and proof of English language proficiency.
    

QUESTION: What is the application deadline?

TOP RETRIEVED DOCUMENT:

    The scholarship application process involves completing an online
    application form and submitting all required documents.

    Applicants should check the official scholarship website for the
    application deadline and any additional requirements.

    Incomplete applications may not be considered.
    

QUESTION: What is the exact scholarship amount?

TOP RETRIEVED DOCUMENT:

    Scholarship recipients may receive financial support toward tuition
    fees and living exp

In [87]:
import gradio as gr

custom_css = """
.gradio-container {
    max-width: 1000px !important;
    margin: auto !important;
    font-family: Arial, sans-serif;
}

footer {
    display: none !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css
) as chat_demo:

    gr.Markdown(
        """
        # 🎓 Mini RAG Scholarship Chatbot
        ### Ask questions about eligibility, applications, and funding.
        """
    )

    gr.ChatInterface(
        fn=chat_response,
        chatbot=gr.Chatbot(
            height=480,
            show_label=False
        ),
        textbox=gr.Textbox(
            placeholder="Ask a scholarship question...",
            label="Your Question",
            lines=2
        ),
        examples=[
            "Who is eligible for the scholarship?",
            "What documents are required?",
            "What is the scholarship deadline?",
            "What is the exact scholarship amount?"
        ]
    )

chat_demo.launch(share=True)

/tmp/ipykernel_1111/3070372601.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://70c129b0becdda26f8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
